# Lambda vs Kappa — Two Ways to Build the Citi Telemetry Pipeline

This notebook builds both **Lambda** and **Kappa** architectures on top of the Citi telemetry learning stack and shows how to explain the tradeoffs clearly in a Staff Data Engineer interview.

## Citi telemetry framing

Citi monitors thousands of API endpoints for:
- latency
- error rate
- throughput
- severity-based alert escalation

We will use:
- **PostgreSQL** as the historical source of truth
- **Kafka** as the real-time event backbone
- **Spark** for batch and stream computation

## Architecture overview

### Lambda architecture

```text
                  +----------------------+
                  |   PostgreSQL metrics |
                  +----------+-----------+
                             |
                             v
                    +------------------+
                    |   Batch Layer    |
                    | Spark historical |
                    | hourly P95 view  |
                    +--------+---------+
                             |
                             v
                    +------------------+
                    |   batch_views    |
                    |   Postgres       |
                    +--------+---------+
                             |
                             |
Fresh telemetry ---> Kafka ->+-> Spark Structured Streaming
                             |   Speed Layer
                             v
                    +------------------+
                    |   speed_view     |
                    |   memory sink    |
                    +--------+---------+
                             |
                             v
                    +------------------+
                    | Serving Layer    |
                    | batch + speed    |
                    +------------------+
```

### Kappa architecture

```text
Fresh telemetry ---> Kafka ---> Spark Structured Streaming ---> Kappa aggregate
                         ^                |
                         |                v
                         +------ replay from Kafka
```

**Mental model**
- **Lambda** = one batch path + one speed path + one serving merge
- **Kappa** = one streaming path; replay the log when reprocessing is needed


In [1]:
import os, asyncio
# Local Spark — JRE 8 + winutils (avoids JDK-17 Netty and Windows NativeIO issues)
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
os.environ['JAVA_HOME']         = 'C:/Program Files/Java/jre1.8.0_481'
os.environ['HADOOP_HOME']       = 'C:/winutils'
os.environ['PYSPARK_PYTHON']    = 'C:/py_venv/proj_educate/Scripts/python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'C:/py_venv/proj_educate/Scripts/python.exe'
os.environ['PATH']              = 'C:/winutils/bin;' + os.environ.get('PATH','')
SPARK_MASTER      = 'local[1]'
PG_JDBC_URL       = 'jdbc:postgresql://localhost:5432/de_telemetry'
PG_USER           = 'de_admin'
PG_PASS           = 'DeAdmin2026!'
KAFKA_BOOTSTRAP   = 'localhost:9092'
DRIVER_CLASSPATH  = r'C:/Users/shareuser/.ivy2/jars/org.postgresql_postgresql-42.7.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.spark_spark-sql-kafka-0-10_2.12-3.5.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.spark_spark-token-provider-kafka-0-10_2.12-3.5.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.kafka_kafka-clients-3.4.1.jar;C:/Users/shareuser/.ivy2/jars/org.lz4_lz4-java-1.8.0.jar;C:/Users/shareuser/.ivy2/jars/org.xerial.snappy_snappy-java-1.1.10.5.jar;C:/Users/shareuser/.ivy2/jars/org.apache.commons_commons-pool2-2.11.1.jar'
print('JAVA_HOME:', os.environ['JAVA_HOME'])
print('HADOOP_HOME:', os.environ['HADOOP_HOME'])


JAVA_HOME: C:/Program Files/Java/jre1.8.0_481
HADOOP_HOME: C:/winutils


In [2]:
import os, json, time, uuid
from datetime import datetime, timedelta, timezone
import psycopg2
from psycopg2.extras import execute_values
from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

PG_CONFIG = {'host':'localhost','port':5432,'dbname':'de_telemetry','user':'de_admin','password':'DeAdmin2026!'}
LAMBDA_TOPIC = 'citi.lambda.speed'
KAPPA_TOPIC  = 'citi.kappa.stream'

def get_pg_conn(): return psycopg2.connect(**PG_CONFIG)

def ensure_kafka_topic(name, partitions=1, rf=1):
    admin = AdminClient({'bootstrap.servers': KAFKA_BOOTSTRAP})
    existing = admin.list_topics(timeout=10).topics
    if name not in existing:
        admin.create_topics([NewTopic(name,num_partitions=partitions,replication_factor=rf)])

spark = (SparkSession.builder.appName('citi_lambda_kappa')
    .master(SPARK_MASTER)
    .config('spark.driver.extraClassPath', DRIVER_CLASSPATH)
    .config('spark.sql.shuffle.partitions', '1')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)
print('Kafka bootstrap:', KAFKA_BOOTSTRAP)
print('Postgres db:', PG_CONFIG['dbname'])


Spark version: 3.5.4
Kafka bootstrap: localhost:9092
Postgres db: de_telemetry


In [3]:
# Lambda — Batch Layer
# Load via JDBC (avoids Python-worker serialization on Windows)

batch_df = (
    spark.read.format("jdbc")
    .option("url", PG_JDBC_URL)
    .option("user", "de_admin")
    .option("password", "DeAdmin2026!")
    .option("query",
        "SELECT m.endpoint_id, e.region, m.metric_name, m.value, m.timestamp "
        "FROM metrics m JOIN endpoints e ON m.endpoint_id = e.endpoint_id "
        "WHERE m.metric_name = 'latency_ms'")
    .load()
)

batch_view_df = (
    batch_df
    .withColumn("hour_bucket", F.date_trunc("hour", F.col("timestamp")))
    .groupBy("hour_bucket", "region")
    .agg(F.expr("percentile_approx(value, 0.95, 10000)").alias("p95_latency"))
    .orderBy("hour_bucket", "region")
)

batch_output = [(r["hour_bucket"], r["region"], float(r["p95_latency"]), "batch")
                for r in batch_view_df.collect()]
batch_count = len(batch_output)
print(f"Lambda batch layer: computed {batch_count} hourly region rows")
print("Sample:", batch_output[:5])

# Write batch view to Postgres
import psycopg2
from psycopg2.extras import execute_values
with get_pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS batch_views;")
        cur.execute("""
            CREATE TABLE batch_views (
                hour_bucket TIMESTAMPTZ NOT NULL,
                region VARCHAR(100) NOT NULL,
                p95_latency DOUBLE PRECISION NOT NULL,
                layer VARCHAR(20) NOT NULL
            );""")
        execute_values(cur,
            "INSERT INTO batch_views (hour_bucket, region, p95_latency, layer) VALUES %s",
            batch_output, page_size=500)
    conn.commit()
print("batch_views written:", batch_count, "rows")


Lambda batch layer: computed 0 hourly region rows
Sample: []
batch_views written: 0 rows


In [4]:
# Prepare fresh telemetry events for both Lambda speed and Kappa stream.
# We intentionally produce the same 50 events to both topics so the comparison is apples-to-apples.

ensure_kafka_topic(LAMBDA_TOPIC)
ensure_kafka_topic(KAPPA_TOPIC)

with get_pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT endpoint_id, region
            FROM endpoints
            ORDER BY endpoint_id
            LIMIT 50
        """)
        endpoint_seed = cur.fetchall()

producer = Producer({"bootstrap.servers": KAFKA_BOOTSTRAP})
now_utc = datetime.now(timezone.utc).replace(microsecond=0)

fresh_events = []
for i, (endpoint_id, region) in enumerate(endpoint_seed, start=1):
    event_ts = now_utc + timedelta(seconds=i)
    event = {
        "endpoint_id": int(endpoint_id),
        "region": region,
        "metric_name": "latency_ms",
        "value": float(70 + (i % 10) * 15 + (i * 1.3)),
        "timestamp": event_ts.isoformat(),
    }
    fresh_events.append(event)

def delivery_report(err, msg):
    if err is not None:
        print("Delivery failed:", err)

for event in fresh_events:
    payload = json.dumps(event).encode("utf-8")
    producer.produce(LAMBDA_TOPIC, key=str(event["endpoint_id"]).encode("utf-8"), value=payload, callback=delivery_report)
    producer.produce(KAPPA_TOPIC, key=str(event["endpoint_id"]).encode("utf-8"), value=payload, callback=delivery_report)

producer.flush()

print(f"Produced {len(fresh_events)} events to {LAMBDA_TOPIC}")
print(f"Produced {len(fresh_events)} events to {KAPPA_TOPIC}")
print("First event:", fresh_events[0])


Produced 50 events to citi.lambda.speed
Produced 50 events to citi.kappa.stream
First event: {'endpoint_id': 1, 'region': 'SNG1', 'metric_name': 'latency_ms', 'value': 86.3, 'timestamp': '2026-04-02T00:50:44+00:00'}


In [5]:
# Lambda — Speed Layer
#
# Consume 50 fresh metrics from Kafka, aggregate in Spark Structured Streaming,
# write to memory sink "speed_view", and show results after 8 seconds.

event_schema = T.StructType([
    T.StructField("endpoint_id", T.IntegerType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("metric_name", T.StringType(), True),
    T.StructField("value", T.DoubleType(), True),
    T.StructField("timestamp", T.StringType(), True),
])

lambda_stream_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", LAMBDA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

lambda_events = (
    lambda_stream_raw
    .select(F.from_json(F.col("value").cast("string"), event_schema).alias("e"))
    .select("e.*")
    .withColumn("event_ts", F.to_timestamp("timestamp"))
    .drop("timestamp")
)

lambda_speed_agg = (
    lambda_events
    .withWatermark("event_ts", "1 minute")
    .groupBy(
        F.window("event_ts", "1 hour").alias("hour_window"),
        F.col("region")
    )
    .agg(F.expr("percentile_approx(value, 0.95, 10000)").alias("p95_latency"))
    .select(
        F.col("hour_window.start").alias("hour_bucket"),
        F.col("region"),
        F.col("p95_latency"),
        F.lit("speed").alias("layer")
    )
)

lambda_query = (
    lambda_speed_agg
    .writeStream
    .format("memory")
    .queryName("speed_view")
    .outputMode("complete")
    .trigger(processingTime="2 seconds")
    .start()
)

lambda_query.awaitTermination(8)
lambda_query.stop()

speed_results_df = spark.sql("SELECT * FROM speed_view ORDER BY hour_bucket, region")
speed_results = [tuple(r) for r in speed_results_df.collect()]
print(f"Lambda speed layer: {len(speed_results)} aggregated rows in memory sink")
print("Lambda speed sample:", speed_results[:10])


Lambda speed layer: 12 aggregated rows in memory sink
Lambda speed sample: [(datetime.datetime(2026, 4, 1, 10, 0), 'LON1', 268.7, 'speed'), (datetime.datetime(2026, 4, 1, 10, 0), 'NYC1', 255.7, 'speed'), (datetime.datetime(2026, 4, 1, 10, 0), 'NYC2', 219.8, 'speed'), (datetime.datetime(2026, 4, 1, 10, 0), 'SNG1', 229.7, 'speed'), (datetime.datetime(2026, 4, 1, 15, 0), 'LON1', 268.7, 'speed'), (datetime.datetime(2026, 4, 1, 15, 0), 'NYC1', 255.7, 'speed'), (datetime.datetime(2026, 4, 1, 15, 0), 'NYC2', 219.8, 'speed'), (datetime.datetime(2026, 4, 1, 15, 0), 'SNG1', 229.7, 'speed'), (datetime.datetime(2026, 4, 1, 19, 0), 'LON1', 268.7, 'speed'), (datetime.datetime(2026, 4, 1, 19, 0), 'NYC1', 255.7, 'speed')]


In [6]:
# Lambda — Serving Layer
#
# Merge batch_views + speed_view using a UNION query in psycopg2 and print the merged row count.

speed_rows_for_pg = [
    (r["hour_bucket"], r["region"], float(r["p95_latency"]), r["layer"])
    for r in speed_results_df.collect()
]

with get_pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS lambda_speed_view")
        cur.execute("""
            CREATE TABLE lambda_speed_view (
                hour_bucket TIMESTAMPTZ NOT NULL,
                region VARCHAR(100) NOT NULL,
                p95_latency DOUBLE PRECISION NOT NULL,
                layer VARCHAR(20) NOT NULL
            )
        """)
        if speed_rows_for_pg:
            execute_values(
                cur,
                """
                INSERT INTO lambda_speed_view (hour_bucket, region, p95_latency, layer)
                VALUES %s
                """,
                speed_rows_for_pg
            )

        cur.execute("""
            SELECT COUNT(*)
            FROM (
                SELECT hour_bucket, region, p95_latency, layer FROM batch_views
                UNION ALL
                SELECT hour_bucket, region, p95_latency, layer FROM lambda_speed_view
            ) merged
        """)
        lambda_merged_count = cur.fetchone()[0]

        cur.execute("""
            SELECT hour_bucket, region, p95_latency, layer
            FROM (
                SELECT hour_bucket, region, p95_latency, layer FROM batch_views
                UNION ALL
                SELECT hour_bucket, region, p95_latency, layer FROM lambda_speed_view
            ) merged
            ORDER BY hour_bucket DESC, region
            LIMIT 10
        """)
        lambda_merged_sample = cur.fetchall()

print(f"Lambda serving layer: batch + speed merged — {lambda_merged_count} rows")
print("Serving sample:", lambda_merged_sample)


Lambda serving layer: batch + speed merged — 12 rows
Serving sample: [(datetime.datetime(2026, 4, 1, 19, 0, tzinfo=datetime.timezone.utc), 'LON1', 268.7, 'speed'), (datetime.datetime(2026, 4, 1, 19, 0, tzinfo=datetime.timezone.utc), 'NYC1', 255.7, 'speed'), (datetime.datetime(2026, 4, 1, 19, 0, tzinfo=datetime.timezone.utc), 'NYC2', 219.8, 'speed'), (datetime.datetime(2026, 4, 1, 19, 0, tzinfo=datetime.timezone.utc), 'SNG1', 229.7, 'speed'), (datetime.datetime(2026, 4, 1, 15, 0, tzinfo=datetime.timezone.utc), 'LON1', 268.7, 'speed'), (datetime.datetime(2026, 4, 1, 15, 0, tzinfo=datetime.timezone.utc), 'NYC1', 255.7, 'speed'), (datetime.datetime(2026, 4, 1, 15, 0, tzinfo=datetime.timezone.utc), 'NYC2', 219.8, 'speed'), (datetime.datetime(2026, 4, 1, 15, 0, tzinfo=datetime.timezone.utc), 'SNG1', 229.7, 'speed'), (datetime.datetime(2026, 4, 1, 10, 0, tzinfo=datetime.timezone.utc), 'LON1', 268.7, 'speed'), (datetime.datetime(2026, 4, 1, 10, 0, tzinfo=datetime.timezone.utc), 'NYC1', 255.7, 

In [7]:
# Kappa — Single Stream
#
# Consume the same 50 events from the Kappa topic and compute the same hourly P95
# with one streaming path only.

kappa_stream_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAPPA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

kappa_events = (
    kappa_stream_raw
    .select(F.from_json(F.col("value").cast("string"), event_schema).alias("e"))
    .select("e.*")
    .withColumn("event_ts", F.to_timestamp("timestamp"))
    .drop("timestamp")
)

kappa_agg = (
    kappa_events
    .withWatermark("event_ts", "1 minute")
    .groupBy(
        F.window("event_ts", "1 hour").alias("hour_window"),
        F.col("region")
    )
    .agg(F.expr("percentile_approx(value, 0.95, 10000)").alias("p95_latency"))
    .select(
        F.col("hour_window.start").alias("hour_bucket"),
        F.col("region"),
        F.col("p95_latency")
    )
)

kappa_query = (
    kappa_agg
    .writeStream
    .format("memory")
    .queryName("kappa_view")
    .outputMode("complete")
    .trigger(processingTime="2 seconds")
    .start()
)

kappa_query.awaitTermination(8)
kappa_query.stop()

kappa_results_df = spark.sql("SELECT * FROM kappa_view ORDER BY hour_bucket, region")
kappa_results = [tuple(r) for r in kappa_results_df.collect()]
print(f"Kappa result: {len(kappa_results)} rows from single stream")
print("Kappa sample:", kappa_results[:10])


Kappa result: 12 rows from single stream
Kappa sample: [(datetime.datetime(2026, 4, 1, 10, 0), 'LON1', 268.7), (datetime.datetime(2026, 4, 1, 10, 0), 'NYC1', 255.7), (datetime.datetime(2026, 4, 1, 10, 0), 'NYC2', 219.8), (datetime.datetime(2026, 4, 1, 10, 0), 'SNG1', 229.7), (datetime.datetime(2026, 4, 1, 15, 0), 'LON1', 268.7), (datetime.datetime(2026, 4, 1, 15, 0), 'NYC1', 255.7), (datetime.datetime(2026, 4, 1, 15, 0), 'NYC2', 219.8), (datetime.datetime(2026, 4, 1, 15, 0), 'SNG1', 229.7), (datetime.datetime(2026, 4, 1, 19, 0), 'LON1', 268.7), (datetime.datetime(2026, 4, 1, 19, 0), 'NYC1', 255.7)]


## Tradeoff Comparison

| Dimension | Lambda | Kappa |
|---|---|---|
| Core shape | Batch path + speed path + serving merge | Single streaming path |
| Operational complexity | Higher because you maintain two computation paths and reconcile outputs | Lower because there is one code path |
| Latency | Good, but the batch layer is naturally slower | Excellent for near-real-time use cases |
| Reprocessing model | Rebuild batch history and reconcile with speed layer | Replay the event log and recompute |
| Data consistency risk | Higher because batch and speed outputs can drift | Lower because all results come from one pipeline |
| Cost profile | Higher ops overhead, sometimes justified for heavy historical recompute | Usually simpler and cheaper once streaming is mature |
| Best fit | Mixed estate, legacy batch platform, strict backfill requirements | Event-driven systems with strong streaming backbone |
| Citi framing | Useful during transition periods when batch remained dominant | Better fit once Spark Structured Streaming matured and Kafka replay became reliable |

**Interview framing**

A crisp Staff-level answer is:

> Lambda solves for both correctness and low latency by combining a historical batch path with a real-time speed path, but it introduces duplicate logic and operational overhead. Kappa simplifies the system by treating the stream as the system of record and using replay for reprocessing. Citi moved from Lambda toward Kappa as Spark Structured Streaming matured — reprocessing is replay, not rebuild.


## What Just Happened

### 1) What is Lambda architecture?

Lambda architecture splits processing into two paths:
- a **batch layer** for full historical recomputation
- a **speed layer** for low-latency incremental results

Then a **serving layer** merges both.  
Use it when you need strong historical recompute and your streaming stack is not yet sufficient to do everything cleanly in one path.

### 2) When would you choose Kappa?

Choose Kappa when:
- Kafka is durable and replayable
- your streaming engine is mature
- you want one code path instead of two
- low latency matters more than maintaining separate batch infrastructure

That usually means less code duplication, less drift risk, and easier operations.

### 3) How does Citi's telemetry fit each model?

For Citi telemetry:
- **Lambda** fits when you want historical hourly recompute over 500K metrics while also overlaying fresh Kafka telemetry in seconds.
- **Kappa** fits when Kafka becomes the durable event backbone and Spark Structured Streaming can compute the exact same aggregates directly from the stream.

In interview language:  
**Lambda is safer during transition. Kappa is cleaner once the streaming platform is mature.**
